In [3]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import KFold, ParameterGrid

import sys, os
sys.path.append(os.path.abspath(".."))

from src.quantifiers.baselines import CC, ACC, PACC, PCC
from src.data.bags import generate_bags

In [6]:
bag_sizes = [100, 200, 500]

aggregative_methods = {
  "CC": CC,
  "ACC": ACC,
  "PACC": PACC,
  "PCC": PCC
}

# SVM parameters to search
param_grid = {
  "C": [0.1, 1, 10],
  "kernel": ["linear", "rbf", "poly"],
  "gamma": ["scale", "auto"]
}

In [7]:
data = np.load("./data/processed/mitdb_preprocessed.npz")

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
rec_train = data["rec_train"]
rec_test = data["rec_test"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train positives:", np.sum(y_train==1))
print("Test positives:", np.sum(y_test==1))

Train shape: (77518, 360)
Test shape: (35129, 360)
Train positives: 46054
Test positives: 28998


In [8]:
results = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
for method_name, QClass in aggregative_methods.items():
  print(f"\n=== Método: {method_name} ===")
  for params in ParameterGrid(param_grid):
    print(f"\n--> Testando parâmetros SVM: {params}")
    fold_id = 0

    for train_idx, val_idx in kf.split(X_train):
      X_tr, y_tr = X_train[train_idx], y_train[train_idx]
      X_val, y_val = X_train[val_idx], y_train[val_idx]

      clf = SVC(probability=True, **params)

      quant = QClass(classifier=clf)
      quant.fit(X_tr, y_tr)

      for bag_size in bag_sizes:
        bags_X, bags_prev = generate_bags(
          X_val, y_val,
          bag_size=bag_size,
          n_bags=50,
          random_state=fold_id + bag_size
        )

        for b_id, (bag, true_prev) in enumerate(zip(bags_X, bags_prev)):
          pred_prev = quant.predict(bag)
          ae = abs(pred_prev - true_prev)
          results.append({
            "method": method_name,
            "classifier": "SVM",
            "params": str(params),
            "bag_size": bag_size,
            "fold": fold_id,
            "bag_id": b_id,
            "true_prev": true_prev,
            "pred_prev": pred_prev,
            "abs_error": ae
          })
      
      fold_id += 1


=== Método: CC ===

--> Testando parâmetros SVM: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}


In [ ]:
df_results = pd.DataFrame(results)
df_results.to_csv("aggregative_ablation.csv", index=False)
df_results.head()